# Olist Delivery Insights — Bronze → Silver → Gold (PySpark / Microsoft Fabric)

Run this notebook **once** in a Fabric Lakehouse (free 60-day trial) to capture the data-engineering proof.
It lands the 9 Olist CSVs as raw Delta (**bronze**), cleans and conforms them (**silver**), and builds the
dimensional star schema (**gold**) — the same grain and columns as the Import-mode TMDL model in `/report`.

**Prereq:** create a Lakehouse, upload the 9 CSVs to its `Files/olist/` folder, attach this notebook to it.
Gold Delta tables can then back a Direct Lake (or Import) semantic model.


## 0 · Setup


In [ ]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import IntegerType, DoubleType

# Lakehouse-relative paths (Fabric mounts the attached Lakehouse).
RAW   = 'Files/olist'        # folder you uploaded the CSVs to
BRONZE = 'Tables/bronze_{}'
# Silver and Gold are written as managed Delta tables (Tables/...)

CSVS = {
    'orders'      : 'olist_orders_dataset.csv',
    'order_items' : 'olist_order_items_dataset.csv',
    'reviews'     : 'olist_order_reviews_dataset.csv',
    'customers'   : 'olist_customers_dataset.csv',
    'products'    : 'olist_products_dataset.csv',
    'sellers'     : 'olist_sellers_dataset.csv',
    'category_tx' : 'product_category_name_translation.csv',
}


## 1 · Bronze — land raw CSVs as Delta (verbatim, replayable)


In [ ]:
def land_bronze(name, filename):
    df = (spark.read.format('csv')
             .option('header', True)
             .option('multiLine', True)      # review comments can contain newlines
             .option('escape', '"')
             .load(f'{RAW}/{filename}'))
    (df.write.format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'bronze_{name}'))
    print(f'bronze_{name}: {df.count():,} rows')

for name, filename in CSVS.items():
    land_bronze(name, filename)


## 2 · Silver — clean, type, conform, dedupe

Cast types, parse timestamps, keep the **latest review per order**, join the English category name.


In [ ]:
# Orders: typed timestamps
orders = (spark.table('bronze_orders')
    .withColumn('order_purchase_timestamp', F.to_timestamp('order_purchase_timestamp'))
    .withColumn('order_delivered_customer_date', F.to_timestamp('order_delivered_customer_date'))
    .withColumn('order_estimated_delivery_date', F.to_timestamp('order_estimated_delivery_date')))
orders.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_orders')

# Reviews: latest per order
w = Window.partitionBy('order_id').orderBy(F.col('review_creation_date').desc())
reviews = (spark.table('bronze_reviews')
    .withColumn('review_score', F.col('review_score').cast(IntegerType()))
    .withColumn('review_creation_date', F.to_timestamp('review_creation_date'))
    .withColumn('rn', F.row_number().over(w)).filter('rn = 1')
    .select('order_id', 'review_score'))
reviews.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_reviews')

# Products + English category (title-cased, underscores -> spaces)
prod = spark.table('bronze_products')
tx   = spark.table('bronze_category_tx')
products = (prod.join(tx, 'product_category_name', 'left')
    .withColumn('cat_raw', F.coalesce(F.col('product_category_name_english'),
                                      F.col('product_category_name'), F.lit('unknown')))
    .withColumn('Category', F.initcap(F.regexp_replace('cat_raw', '_', ' ')))
    .select('product_id', 'Category'))
products.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_products')

# Customers / Sellers (typed pass-through)
spark.table('bronze_customers').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_customers')
spark.table('bronze_sellers').write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_sellers')
spark.table('bronze_order_items').withColumn('price', F.col('price').cast(DoubleType()))\
    .withColumn('freight_value', F.col('freight_value').cast(DoubleType()))\
    .write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_order_items')
print('silver layer built')


## 3 · Gold — dimensional star schema

Same tables, grains and columns as the TMDL model in `/report`.


In [ ]:
# ---- Dimensions ----
Dim_Customer = spark.table('silver_customers').select(
    'customer_id', 'customer_unique_id',
    F.col('customer_city').alias('Customer City'),
    F.col('customer_state').alias('Customer State'))

Dim_Product = spark.table('silver_products')        # product_id, Category
Dim_Seller  = spark.table('silver_sellers').select(
    'seller_id', F.col('seller_city').alias('Seller City'), F.col('seller_state').alias('Seller State'))

# Dim_Date: one contiguous row per day across the order history
bounds = spark.table('silver_orders').agg(
    F.year(F.min('order_purchase_timestamp')).alias('y0'),
    F.year(F.max('order_purchase_timestamp')).alias('y1')).first()
Dim_Date = (spark.sql(f"SELECT explode(sequence(to_date('{bounds.y0}-01-01'), to_date('{bounds.y1}-12-31'), interval 1 day)) AS Date")
    .withColumn('Year', F.year('Date'))
    .withColumn('Quarter', F.concat(F.lit('Q'), F.quarter('Date')))
    .withColumn('Month Number', F.month('Date'))
    .withColumn('Month', F.date_format('Date', 'MMM'))
    .withColumn('Month Year', F.date_format('Date', 'MMM yyyy'))
    .withColumn('Month Year Sort', F.year('Date') * 100 + F.month('Date')))


In [ ]:
# ---- Facts ----
o = spark.table('silver_orders')
Fact_Orders = (o.join(spark.table('silver_reviews'), 'order_id', 'left')
    .withColumn('purchase_date', F.to_date('order_purchase_timestamp'))
    .withColumn('delivery_days', F.when(F.col('order_delivered_customer_date').isNotNull(),
        F.datediff('order_delivered_customer_date', 'order_purchase_timestamp')))
    .withColumn('is_late', F.when(
        F.col('order_delivered_customer_date').isNotNull() & F.col('order_estimated_delivery_date').isNotNull(),
        F.col('order_delivered_customer_date') > F.col('order_estimated_delivery_date')))
    .select('order_id', 'customer_id', 'purchase_date',
            F.col('order_status').alias('Order Status'), 'review_score', 'delivery_days', 'is_late',
            F.to_date('order_delivered_customer_date').alias('Delivered Date'),
            F.to_date('order_estimated_delivery_date').alias('Estimated Date')))

# Line fact: denormalise customer_id + purchase_date from the order header
hdr = o.select('order_id', 'customer_id', F.to_date('order_purchase_timestamp').alias('purchase_date'))
Fact_OrderItems = (spark.table('silver_order_items')
    .join(hdr, 'order_id', 'inner')
    .select('order_id', 'product_id', 'seller_id', 'customer_id', 'purchase_date', 'price', 'freight_value'))

# ---- Write gold Delta tables ----
for tname, df in [('Dim_Date', Dim_Date), ('Dim_Customer', Dim_Customer), ('Dim_Product', Dim_Product),
                  ('Dim_Seller', Dim_Seller), ('Fact_Orders', Fact_Orders), ('Fact_OrderItems', Fact_OrderItems)]:
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'gold_{tname}')
    print(f'gold_{tname}: {df.count():,} rows')


## 4 · Quick validation — Revenue at Risk (cross-grain)

Confirms the same headline number the DAX measure produces: line-level sales for orders flagged late or rated ≤ 2.


In [ ]:
risky = spark.table('gold_Fact_Orders').filter('is_late = true OR review_score <= 2').select('order_id')
items = spark.table('gold_Fact_OrderItems')
total = items.agg(F.sum('price')).first()[0]
at_risk = items.join(risky, 'order_id', 'inner').agg(F.sum('price')).first()[0]
print(f'Total Sales      : R$ {total:,.0f}')
print(f'Revenue at Risk  : R$ {at_risk:,.0f}  ({100*at_risk/total:.1f}% of sales)')
